In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


clean_results_path = "../clean_results/greedy/results.json"
clean_results = pd.read_json(clean_results_path, orient="records")

dirty_results_path = "../logs/silent-norm-final-v1/results.json"
dirty_results = pd.read_json(dirty_results_path, orient="records")

In [ ]:
RELEVANT_FILES = [
    "anli",
    "blimp",
    "mastermind_easy",
    "metabench_arc",
    "metabench_hellaswag",
    "metabench_mmlu",
    "metabench_truthfulqa",
    "metabench_winogrande",
    "piqa",
    "toxigen",
    "wmdp",
]

In [ ]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring

    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    return out


clean_results = filter_files(clean_results, RELEVANT_FILES)
dirty_results = filter_files(dirty_results, RELEVANT_FILES)

In [ ]:
def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out


def add_dirty_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


clean_results = add_clean_model_name(clean_results)
dirty_results = add_dirty_path_metadata(dirty_results)

In [ ]:
def consolidate_blimp_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    # consolidates all benchmarks whose name begin with blimp_ into a single blimp benchmark
    # the score of blimp is weighted average of the scores

    df = df.copy()
    df = df[df["benchmark"] != "blimp"]

    # rename the benchmark for everyone to blimp
    # sample_index should be from 0 to len(blimp_df) - 1 (redo it)

    blimp_df = df[df["benchmark"].str.startswith("blimp_")].copy()
    blimp_df["benchmark"] = "blimp"
    blimp_df["sample_index"] = range(len(blimp_df))

    # concat

    df = pd.concat([df.reset_index(drop=True), blimp_df.reset_index(drop=True)], ignore_index=True)
    df = df.reset_index(drop=True)

    return df

In [ ]:
clean_results = consolidate_blimp_benchmark(clean_results)
dirty_results = consolidate_blimp_benchmark(dirty_results)

In [ ]:
def compute_logprobs_normalized(df: pd.DataFrame) -> pd.DataFrame:
    # computes logprobs_normalized
    # each entry in logprobs column contains list[float]
    # each entry in num_tokens contains list[int]
    # logprobs_normalized = logprob[i] / num_tokens[i]

    out = df.copy()
    out["logprobs_normalized"] = out.apply(
        lambda row: [lp / nt if nt > 0 else float("nan") for lp, nt in zip(row["logprobs"], row["choice_lengths"])], axis=1
    )
    return out


clean_results = compute_logprobs_normalized(clean_results)
dirty_results = compute_logprobs_normalized(dirty_results)

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import poisson_binom
from scipy.stats import rv_discrete

def build_gt_distributions(
    df: pd.DataFrame,
    *,
    index_columns: list[str],
    logprobs_col: str = "logprobs",
    gold_index_col: str = "gold_index",
    sample_index_col: str = "sample_index",
):
    """
    Returns a dict:
        key (tuple of index_columns values) -> {
            "n": int,
            "mean": float,
            "var": float,
            "std": float,
            "dist": poisson_binom object
        }
    """

    def gold_prob(logprobs, gold_index):
        arr = np.asarray(logprobs, dtype=np.float64)
        return float(np.exp(arr[gold_index] - logsumexp(arr)))

    df = df.copy()
    df["_p"] = [gold_prob(lp, gi) for lp, gi in zip(df[logprobs_col], df[gold_index_col])]

    out = {}

    grouped = df.groupby(index_columns, sort=False)

    for key, g in grouped:
        # ensure key is always a tuple for consistency
        if not isinstance(key, tuple):
            key = (key,)

        g = g.sort_values(sample_index_col)

        p = g["_p"].to_numpy()
        n = len(p)

        mean = float(np.mean(p))
        var = float(np.sum(p * (1 - p)) / (n * n))
        std = float(np.sqrt(var))

        out[key] = {
            "n": n,
            "mean": mean,
            "var": var,
            "std": std,
            "dist": poisson_binom(p),
        }

    return out


def upper_tail(d: dict, s: float | np.ndarray):
    n = d["n"]
    
    s_arr = np.asarray(s, dtype=np.float64)
    ks = np.ceil(s_arr * n).astype(int)
    
    results = np.empty_like(ks, dtype=np.float64)
    results[ks <= 0] = 1.0
    results[ks > n] = 0.0
    mask = (ks > 0) & (ks <= n)
    
    dist: rv_discrete = d["dist"]
    results[mask] = dist.sf(ks[mask] - 1)  # P(X >= k)
    return results if isinstance(s, np.ndarray) else float(results[0])


def lower_tail(d: dict, s: float | np.ndarray):
    n = d["n"]
    s_arr = np.asarray(s, dtype=np.float64)
    ks = np.floor(s_arr * n).astype(int)
    results = np.empty_like(ks, dtype=np.float64)
    results[ks < 0] = 0.0
    results[ks >= n] = 1.0
    mask = (ks >= 0) & (ks < n)
    
    dist: rv_discrete = d["dist"]
    results[mask] = dist.cdf(ks[mask])  # P(X <= k)
    return results if isinstance(s, np.ndarray) else float(results[0])


def two_sided_tail(d: dict, s: float | np.ndarray):
    mean = d["mean"]
    s_arr = np.asarray(s, dtype=np.float64)
    delta = np.abs(s_arr - mean)
    
    lower = lower_tail(d, mean - delta)
    upper = upper_tail(d, mean + delta)
    results = np.minimum(1.0, lower + upper)
    return results if isinstance(s, np.ndarray) else float(results[0])

In [ ]:
import numpy as np


def expected_tails(d_gt: dict, d_alt: dict) -> tuple[float, float, float]:
    """
    Exact expectation of 
    - two_sided_tail(d_gt, s) 
    - lower_tail(d_gt,  m - |m-s|)
    - upper_tail(d_gt, m + |m-s|)
    
    where s is sampled from the score distribution induced by d_alt,
    and m = d_gt["mean"] is the mean score under the ground truth distribution.

    Here d_alt["dist"] is a Poisson-binomial distribution over counts Y,
    and the score is s = Y / d_alt["n"].

    Returns:
        - E_{s ~ d_alt}[ two_sided_tail(d_gt, s) ]
        - E_{s ~ d_alt}[ lower_tail(d_gt, m - |m-s|) ]
        - E_{s ~ d_alt}[ upper_tail(d_gt, m + |m-s|) ]
    """
    n_alt = d_alt["n"]
    ks = np.arange(n_alt + 1)

    # exact PMF of the alternative count distribution
    pmf = d_alt["dist"].pmf(ks)
    pmf = np.asarray(pmf, dtype=np.float64)

    # optional renormalization for numerical stability
    pmf_sum = pmf.sum()
    if pmf_sum <= 0:
        raise ValueError("Alternative PMF is numerically zero / invalid.")
    pmf = pmf / pmf_sum

    # exact tail statistic for each possible realized score s = k / n_alt
    two_sided, lower, upper = two_sided_tail(d_gt, ks / n_alt, return_extra=True)
    
    return float(np.dot(pmf, two_sided)), float(np.dot(pmf, lower)), float(np.dot(pmf, upper))

In [ ]:
# Build ground truth distributions for both metrics from clean results

clean_index = ["benchmark", "model_name"]

print("Building clean distributions for acc...")
dist_acc = build_gt_distributions(
    clean_results,
    index_columns=clean_index,
    logprobs_col="logprobs",
)

print("Building clean distributions for acc_norm...")
dist_acc_norm = build_gt_distributions(
    clean_results,
    index_columns=clean_index,
    logprobs_col="logprobs_normalized",
)

In [ ]:
# Build modified distributions for both metrics from dirty results

dirty_index = ["benchmark", "model_name", "layer_name", "exp_name"]

print("Building dirty distributions for acc...")
dist_dirty_acc = build_gt_distributions(
    dirty_results,
    index_columns=dirty_index,
    logprobs_col="logprobs",
)

print("Building dirty distributions for acc_norm...")
dist_dirty_acc_norm = build_gt_distributions(
    dirty_results,
    index_columns=dirty_index,
    logprobs_col="logprobs_normalized",
)

In [ ]:
from tqdm.auto import tqdm

def build_comparison_df(dist_orig: dict, dist_mod: dict, orig_index: list[str], mod_index: list[str]) -> pd.DataFrame:
    """
    Iterates over dirty groups, matching them to clean groups via shared index columns.
    Computes the expected two-sided tail and returns a freshly built DataFrame.
    """
    def compute_original_key(mod_key):
        orig_key = []
        for col in orig_index:
            col_val = mod_key[mod_index.index(col)]
            orig_key.append(col_val)
        return tuple(orig_key)  
    
    
    records = []
    for mod_key, mod_info in tqdm(dist_mod.items()):
        if not isinstance(mod_key, tuple):
            mod_key = (mod_key,)  
        
        orig_key = compute_original_key(mod_key)
        orig_info = dist_orig[orig_key]
        
        tail, lower, upper = expected_tails(orig_info, mod_info)
        
        record = {k: v for k, v in zip(mod_index, mod_key)}
        record.update(
            {
                "clean_mean": orig_info["mean"],
                "dirty_mean": mod_info["mean"],
                "clean_std": orig_info["std"],
                "dirty_std": mod_info["std"],
                "two_sided_tail": tail,
                "lower_tail": lower,
                "upper_tail": upper,
                "count": mod_info["n"],
            }
        )
        
        records.append(record)
        
    return pd.DataFrame(records)

In [ ]:
# remove values with keys with benchmark "blimp" from dist_acc, dist_acc_norm, dist_dirty_acc, dist_dirty_acc_norm

dist_acc_new = {k: v for k, v in dist_acc.items() if "blimp_" not in k[0]}
dist_dirty_acc_new = {k: v for k, v in dist_dirty_acc.items() if "blimp_" not in k[0]}

dist_acc_norm_new = {k: v  for k, v in dist_acc_norm.items() if "blimp_" not in k[0]}
dist_dirty_acc_norm_new = {k: v for k, v in dist_dirty_acc_norm.items() if "blimp_" not in k[0]}

In [ ]:
df_acc_comparison = build_comparison_df(dist_acc_new, dist_dirty_acc_new, clean_index, dirty_index)
df_acc_norm_comparison = build_comparison_df(dist_acc_norm_new, dist_dirty_acc_norm_new, clean_index, dirty_index)

In [ ]:
df_acc_comparison